# Hyperparameter Tuning Model with k-fold Cross Validation

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [1]:
# Import necessary libraries

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from datasets import concatenate_datasets, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader


In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")   # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device("cpu")   # Fallback

print(f"Using Device: {DEVICE}")

Using Device: mps


## Model Choice

[Explain why you've chosen a particular model as the baseline. This could be a simple statistical model or a basic machine learning model. Justify your choice.]


We chose a convolutional neural network (CNN) as the baseline model for our aesthetic emotions map project. CNNs are well-suited for image classification tasks due to their ability to capture spatial hierarchies in images. They can learn to recognize patterns and features in the images that are relevant for predicting the associated emotions. Additionally, CNNs have been widely used and have shown strong performance in various image-related tasks, making them a reasonable starting point for our project.

## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]

For the baseline model, we will be using the raw pixel values of the images as features. This is a common approach for image classification tasks, as it allows the model to learn directly from the visual data without any manual feature engineering. By using the raw pixel values, we can leverage the CNN's ability to automatically extract relevant features from the images during training.

In [3]:
# Loading the dataset using Hugging Face's datasets library
dataset = load_dataset("bjoern-doege/aesthetic-emotions-map")

Resolving data files:   0%|          | 0/4721 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1025 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1089 [00:00<?, ?it/s]

In [4]:
NORMALIZE_MEAN = [0.5111283659934998, 0.48830345273017883, 0.46479079127311707]
NORMALIZE_STD = [0.3433663249015808, 0.3207928538322449, 0.32255250215530396]


In [5]:
# Converting labels to ids
# PyTorch's ImageFolder would do this automatically, but we are using Hugging Face's datasets library
label_names = sorted(dataset["train"].unique("label"))
label_to_id = {label: i for i, label in enumerate(label_names)}

In [6]:
# Use train + validation for cross-validation and keep test untouched for final evaluation.
trainval_dataset = concatenate_datasets([
    dataset["train"],
    dataset["validation"],
])

test_dataset = dataset["test"]

NUM_EPOCHS = 10
N_SPLITS = 7
RANDOM_STATE = 42


In [7]:
X = np.arange(len(trainval_dataset))
y = [label_to_id[label] for label in trainval_dataset["label"]]
groups = trainval_dataset["style"]

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

style_overlap = set(trainval_dataset["style"]) & set(test_dataset["style"])
print(f"Train+validation samples: {len(trainval_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Overlapping styles between train+validation and test: {len(style_overlap)}")


Train+validation samples: 5744
Test samples: 1088
Overlapping styles between train+validation and test: 0


## Implementation

The baseline model is a configurable CNN. Optuna chooses the number of convolutional layers, filters, kernel sizes, dropout rate, fully connected layer size, learning rate, image resolution, and batch size. K-fold cross-validation is used inside each Optuna trial, and the final model is trained once on all train+validation data with the best parameters.


In [8]:
class FlexibleCNN(nn.Module):
    def __init__(self, n_layers, n_filters, kernel_sizes, dropout_rate, fc_size, num_classes):
        super().__init__()

        if len(n_filters) != n_layers or len(kernel_sizes) != n_layers:
            raise ValueError("n_filters and kernel_sizes must match n_layers")

        layers = []
        in_channels = 3

        for out_channels, kernel_size in zip(n_filters, kernel_sizes):
            padding = (kernel_size - 1) // 2
            layers.extend([
                nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2),
            ])
            in_channels = out_channels

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(in_channels, fc_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(fc_size, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [14]:
def design_search_space(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)

    valid_resolutions = [
        resolution
        for resolution in [16, 32, 64]
        if resolution >= 2 ** n_layers
    ]

    return {
        "n_layers": n_layers,
        "n_filters": [
            trial.suggest_int(f"n_filters_layer{i}", 8, 64, step=8)
            for i in range(n_layers)
        ],
        "kernel_sizes": [
            trial.suggest_int(f"kernel_size_layer{i}", 3, 5, step=2)
            for i in range(n_layers)
        ],
        "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.5),
        "fc_size": trial.suggest_int("fc_size", 64, 512, step=64),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "resolution": trial.suggest_categorical("resolution", valid_resolutions),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32, 64]),
    }


def expand_best_params(best_params):
    params = best_params.copy()
    params["n_filters"] = [
        params[f"n_filters_layer{i}"]
        for i in range(params["n_layers"])
    ]
    params["kernel_sizes"] = [
        params[f"kernel_size_layer{i}"]
        for i in range(params["n_layers"])
    ]
    return params


In [10]:
def compute_metrics(y_true, y_pred, total_loss):
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


def training_epoch(model, train_loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    y_true = []
    y_pred = []

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in data_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predicted = outputs.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    return compute_metrics(y_true, y_pred, running_loss)


In [11]:
def create_model(params):
    return FlexibleCNN(
        n_layers=params["n_layers"],
        n_filters=params["n_filters"],
        kernel_sizes=params["kernel_sizes"],
        dropout_rate=params["dropout_rate"],
        fc_size=params["fc_size"],
        num_classes=len(label_names),
    )


def create_optimizer(model, params):
    return optim.Adam(model.parameters(), lr=params["learning_rate"])


def compute_class_weights(ds):
    labels = [label_to_id[label] for label in ds["label"]]
    class_counts = np.bincount(labels, minlength=len(label_names))
    class_counts = np.maximum(class_counts, 1)
    weights = len(labels) / (len(label_names) * class_counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def create_loss_fn(ds):
    class_weights = compute_class_weights(ds)
    return nn.CrossEntropyLoss(weight=class_weights)


def create_transform(resolution, augment=False):
    transform_steps = [transforms.Resize((resolution, resolution))]

    if augment:
        transform_steps.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
        ])

    transform_steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
    ])

    return transforms.Compose(transform_steps)


def create_preprocess(transform):
    def preprocess(examples):
        return {
            "image": [transform(image.convert("RGB")) for image in examples["image"]],
            "label": torch.tensor(
                [label_to_id[label] for label in examples["label"]],
                dtype=torch.long,
            ),
        }

    return preprocess


def create_dataloader(ds, params, shuffle=False, augment=False):
    transform = create_transform(params["resolution"], augment=augment)
    preprocess = create_preprocess(transform)

    return DataLoader(
        ds.with_transform(preprocess),
        batch_size=params["batch_size"],
        shuffle=shuffle,
        num_workers=0,
    )


In [12]:
def objective(trial):
    params = design_search_space(trial)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
        fold_train_dataset = trainval_dataset.select(train_idx.tolist())
        fold_val_dataset = trainval_dataset.select(val_idx.tolist())

        train_loader = create_dataloader(
            fold_train_dataset,
            params,
            shuffle=True,
            augment=True,
        )
        val_loader = create_dataloader(
            fold_val_dataset,
            params,
            shuffle=False,
            augment=False,
        )

        model = create_model(params).to(DEVICE)
        optimizer = create_optimizer(model, params)
        loss_fn = create_loss_fn(fold_train_dataset)

        for epoch in range(NUM_EPOCHS):
            training_epoch(model, train_loader, optimizer, loss_fn, DEVICE)

        val_metrics = evaluate_model(model, val_loader, loss_fn, DEVICE)
        fold_scores.append(val_metrics["f1_macro"])

    return np.mean(fold_scores)


In [15]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best params:")
print(study.best_params)
print(f"Best mean CV macro F1: {study.best_value:.4f}")


[I 2026-06-15 18:50:32,434] A new study created in memory with name: no-name-124172eb-03f1-457a-a8af-4cb13c3317e4
[I 2026-06-15 19:00:30,360] Trial 0 finished with value: 0.2741411960773736 and parameters: {'n_layers': 1, 'n_filters_layer0': 48, 'kernel_size_layer0': 5, 'dropout_rate': 0.35774548045917504, 'fc_size': 448, 'learning_rate': 0.005824353536565335, 'resolution': 64, 'batch_size': 64}. Best is trial 0 with value: 0.2741411960773736.
[I 2026-06-15 19:09:55,910] Trial 1 finished with value: 0.265170063268216 and parameters: {'n_layers': 1, 'n_filters_layer0': 24, 'kernel_size_layer0': 5, 'dropout_rate': 0.17373003179355073, 'fc_size': 512, 'learning_rate': 0.0067625238737405065, 'resolution': 64, 'batch_size': 64}. Best is trial 0 with value: 0.2741411960773736.
[I 2026-06-15 19:24:11,473] Trial 2 finished with value: 0.27580484825368706 and parameters: {'n_layers': 2, 'n_filters_layer0': 8, 'n_filters_layer1': 40, 'kernel_size_layer0': 3, 'kernel_size_layer1': 3, 'dropout_rat

Best params:
{'n_layers': 3, 'n_filters_layer0': 40, 'n_filters_layer1': 40, 'n_filters_layer2': 64, 'kernel_size_layer0': 3, 'kernel_size_layer1': 3, 'kernel_size_layer2': 3, 'dropout_rate': 0.3849246984111697, 'fc_size': 320, 'learning_rate': 0.0005172580112598357, 'resolution': 64, 'batch_size': 8}
Best mean CV macro F1: 0.3197


## Evaluation

Optuna selects hyperparameters by maximizing mean cross-validation macro-F1 on the combined train+validation split. After tuning, the final model is trained on all train+validation data and evaluated once on the untouched test split. Reported metrics include loss, accuracy, weighted precision/recall/F1, macro-F1, a classification report, and a confusion matrix.


In [16]:
# Train one final model with the best Optuna parameters, then evaluate once on the untouched test set.
best_params = expand_best_params(study.best_params)

final_model = create_model(best_params).to(DEVICE)
final_optimizer = create_optimizer(final_model, best_params)
final_loss_fn = create_loss_fn(trainval_dataset)

trainval_loader = create_dataloader(
    trainval_dataset,
    best_params,
    shuffle=True,
    augment=True,
)
test_loader = create_dataloader(
    test_dataset,
    best_params,
    shuffle=False,
    augment=False,
)

for epoch in range(NUM_EPOCHS):
    train_metrics = training_epoch(final_model, trainval_loader, final_optimizer, final_loss_fn, DEVICE)
    print(
        f"Final model epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f}, "
        f"train_acc={train_metrics['accuracy']:.4f}, "
        f"train_macro_f1={train_metrics['f1_macro']:.4f}"
    )

test_metrics = evaluate_model(final_model, test_loader, final_loss_fn, DEVICE)

print("\nFinal test metrics")
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Weighted precision: {test_metrics['precision_weighted']:.4f}")
print(f"Weighted recall: {test_metrics['recall_weighted']:.4f}")
print(f"Weighted F1: {test_metrics['f1_weighted']:.4f}")
print(f"Macro F1: {test_metrics['f1_macro']:.4f}")

print("\nClassification report:")
print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    labels=list(range(len(label_names))),
    target_names=label_names,
    zero_division=0,
))

print("\nConfusion matrix:")
print(confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"]))


Final model epoch 1/10 | train_loss=1.8456, train_acc=0.2725, train_macro_f1=0.2575
Final model epoch 2/10 | train_loss=1.7174, train_acc=0.3139, train_macro_f1=0.3086
Final model epoch 3/10 | train_loss=1.6860, train_acc=0.3303, train_macro_f1=0.3257
Final model epoch 4/10 | train_loss=1.6494, train_acc=0.3478, train_macro_f1=0.3413
Final model epoch 5/10 | train_loss=1.6180, train_acc=0.3564, train_macro_f1=0.3513
Final model epoch 6/10 | train_loss=1.6063, train_acc=0.3602, train_macro_f1=0.3544
Final model epoch 7/10 | train_loss=1.5908, train_acc=0.3700, train_macro_f1=0.3656
Final model epoch 8/10 | train_loss=1.5684, train_acc=0.3698, train_macro_f1=0.3658
Final model epoch 9/10 | train_loss=1.5562, train_acc=0.3835, train_macro_f1=0.3815
Final model epoch 10/10 | train_loss=1.5497, train_acc=0.3795, train_macro_f1=0.3772

Final test metrics
Test loss: 1.6738
Accuracy: 0.3493
Weighted precision: 0.4249
Weighted recall: 0.3493
Weighted F1: 0.3593
Macro F1: 0.3206

Classification 

In [ ]:
# Display a sample of test images with true and predicted labels.
def plot_test_predictions(test_dataset, test_metrics, label_names, num_images=12, seed=RANDOM_STATE):
    num_images = min(num_images, len(test_dataset), len(test_metrics["y_true"]))
    rng = np.random.default_rng(seed)
    sample_indices = rng.choice(len(test_dataset), size=num_images, replace=False)

    n_cols = 4
    n_rows = int(np.ceil(num_images / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4.4 * n_rows))
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, sample_indices):
        image = test_dataset[int(idx)]["image"].convert("RGB")
        true_label = label_names[test_metrics["y_true"][int(idx)]]
        pred_label = label_names[test_metrics["y_pred"][int(idx)]]
        is_correct = true_label == pred_label

        ax.imshow(image)
        ax.set_title(
            f"True: {true_label}\nPred: {pred_label}",
            color="green" if is_correct else "red",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[num_images:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_test_predictions(test_dataset, test_metrics, label_names)


In [17]:
torch.save(final_model.state_dict(), "model_w_k-fold_best_params.pth")
